# 01 · Baseline inference — why a generic detector is not enough
**MAICEN-0526 · M4U3 Computer Vision · Group 7**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solomon8909/maicen0526-m4u3-ppe-detection/blob/main/notebooks/01_baseline_inference.ipynb)

Runs with **no accounts, no API keys and no Colab secrets** in about 3 minutes on CPU or GPU. Everything it needs — the weights and the unseen site images — is downloaded from this repository's public Release and checksum-verified.

- **Part A:** an off-the-shelf YOLOv8n trained on COCO (80 everyday classes) looks at our site photos. It finds *people* but has no concept of a hardhat or a hi-vis vest — so it cannot answer the safety question.
- **Part B:** our fine-tuned PPE model (downloaded from the GitHub Release) looks at the same photos.

`Runtime → Disconnect and delete runtime`, then `Run all`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
from pathlib import Path
import urllib.request, cv2, ultralytics, torch
from ultralytics import YOLO
from IPython.display import Image, display

WEIGHTS_URL       = "https://github.com/solomon8909/maicen0526-m4u3-ppe-detection/releases/download/v1.0/best.pt"
NEW_IMAGES_URL    = "https://github.com/solomon8909/maicen0526-m4u3-ppe-detection/releases/download/v1.0/new-images.zip"
NEW_IMAGES_SHA256 = "5d3597fc72a5e0d2b38a7eeaa969aa80ad6c13df59ad543d9bb93404b1958dde"
CONF = 0.25

import hashlib, zipfile
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

NEW_DIR = Path("new_images")
if not list(NEW_DIR.rglob("*.jpg")) + list(NEW_DIR.rglob("*.png")):
    NEW_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(NEW_IMAGES_URL, "new_images.zip")
    digest = sha256_file("new_images.zip")
    if NEW_IMAGES_SHA256 and not NEW_IMAGES_SHA256.startswith("PASTE") and digest.lower() != NEW_IMAGES_SHA256.strip().lower():
        Path("new_images.zip").unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    print("SHA256 verified:", digest)
    with zipfile.ZipFile("new_images.zip") as z:
        z.extractall(NEW_DIR)
IMGS = sorted(p for p in NEW_DIR.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
OUT = Path("results/baseline"); OUT.mkdir(parents=True, exist_ok=True)
print(f"ultralytics {ultralytics.__version__} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | {len(IMGS)} images")

## Part A · Generic COCO-pretrained YOLOv8n

In [ ]:
coco = YOLO("yolov8n.pt")
for p in IMGS:
    r = coco.predict(str(p), conf=CONF, verbose=False)[0]
    cv2.imwrite(str(OUT / f"coco_{p.stem}.jpg"), r.plot())
    found = sorted({r.names[int(c)] for c in r.boxes.cls.cpu().numpy()})
    print(f"{p.name}: {found}")
print("\nCOCO classes containing 'hat', 'helmet' or 'vest':",
      [n for n in coco.names.values() if any(k in n for k in ("hat", "helmet", "vest"))] or "none")

## Part B · Fine-tuned PPE model (GitHub Release weights)

In [ ]:
Path("weights").mkdir(exist_ok=True)
if not Path("weights/best.pt").exists():
    urllib.request.urlretrieve(WEIGHTS_URL, "weights/best.pt")
ppe = YOLO("weights/best.pt")
print("Classes:", list(ppe.names.values()))
for p in IMGS:
    r = ppe.predict(str(p), conf=CONF, verbose=False)[0]
    cv2.imwrite(str(OUT / f"ppe_{p.stem}.jpg"), r.plot())
    counts = {}
    for c in r.boxes.cls.cpu().numpy(): counts[r.names[int(c)]] = counts.get(r.names[int(c)], 0) + 1
    print(f"{p.name}: {counts}")

## Side by side

In [ ]:
import numpy as np
for p in IMGS[:5]:
    a, b = cv2.imread(str(OUT / f"coco_{p.stem}.jpg")), cv2.imread(str(OUT / f"ppe_{p.stem}.jpg"))
    cv2.imwrite(str(OUT / f"compare_{p.stem}.jpg"), np.hstack([a, np.full((a.shape[0], 8, 3), 255, np.uint8), b]))
    display(Image(str(OUT / f"compare_{p.stem}.jpg"), width=1000))